In [8]:
import pandas as pd
import networkx as nx
import pydot
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.colors as mcolors
import numpy as np
from typing import Optional, List, Tuple, Set # Import Set for type hinting
from matplotlib.transforms import blended_transform_factory
from matplotlib.lines import Line2D

# Use this line to tell your script where to find the 'dot.exe' program
os.environ["PATH"] += os.pathsep + r'C:\Program Files\Graphviz\bin'

# --- Helper Functions for plot_diploid_chromosome_ancestry ---

def get_unique_ancestries(locus_data_df: pd.DataFrame) -> List[Tuple[str, int]]:
    """
    Extracts all unique (ancestral_population, founder_id) pairs from the DataFrame
    using the correct column names.
    """
    unique_ancestries_set = set()

    # Collect ancestries from chromatid1
    temp_df1 = locus_data_df[['ancestry_chromatid1_pop', 'ancestry_chromatid1_founder_id']].drop_duplicates().dropna()
    for _, row in temp_df1.iterrows():
        # Ensure that founder_id is an int (it might be float due to NaNs)
        if pd.notna(row['ancestry_chromatid1_founder_id']):
            unique_ancestries_set.add((str(row['ancestry_chromatid1_pop']), int(row['ancestry_chromatid1_founder_id'])))

    # Collect ancestries from chromatid2
    temp_df2 = locus_data_df[['ancestry_chromatid2_pop', 'ancestry_chromatid2_founder_id']].drop_duplicates().dropna()
    for _, row in temp_df2.iterrows():
        # Ensure that founder_id is an int (it might be float due to NaNs)
        if pd.notna(row['ancestry_chromatid2_founder_id']):
            unique_ancestries_set.add((str(row['ancestry_chromatid2_pop']), int(row['ancestry_chromatid2_founder_id'])))

    return sorted(list(unique_ancestries_set))


def generate_color_map_for_ancestries(unique_ancestries: List[Tuple[str, int]]) -> dict:
    """
    Generates a consistent color map for unique ancestral populations and founder IDs.
    Assigns specific colors to P_A and P_B founders, then uses a colormap for others.
    """
    colors = {}

    # Define a base color for P_A and P_B, and then generate shades for their founders
    parent_base_colors = {
        'P_A': plt.cm.Blues,   # Use a blue colormap for P_A founders
        'P_B': plt.cm.Reds    # Use a red colormap for P_B founders
    }

    # Group ancestries by their population label
    grouped_ancestries = {}
    for anc_pop, founder_id in unique_ancestries:
        grouped_ancestries.setdefault(anc_pop, []).append((anc_pop, founder_id))

    # Assign colors for P_A and P_B founders
    for pop_label, anc_list in grouped_ancestries.items():
        if pop_label in parent_base_colors:
            cmap = parent_base_colors[pop_label]
            # Sort founder IDs to ensure consistent color assignment
            sorted_founders = sorted(anc_list, key=lambda x: x[1])
            num_founders = len(sorted_founders)
            for i, (anc_pop, founder_id) in enumerate(sorted_founders):
                # Use a range within the colormap (e.g., from 0.3 to 0.9 for darker shades)
                color_value = (i + 1) / (num_founders + 1) # Distribute evenly
                colors[(anc_pop, founder_id)] = mcolors.to_hex(cmap(color_value * 0.6 + 0.3)) # Adjust 0.6 and 0.3 for shade range

    # Generate additional colors for other ancestries (e.g., F1, F2 if they appear in your data)
    remaining_ancestries = [anc for anc in unique_ancestries if anc[0] not in parent_base_colors]
    num_remaining = len(remaining_ancestries)

    if num_remaining > 0:
        # Use a different colormap for other ancestries to distinguish them
        cmap_other = plt.cm.get_cmap('Dark2' if num_remaining <= 8 else 'tab20' if num_remaining <= 20 else 'hsv', num_remaining)
        for i, ancestry in enumerate(remaining_ancestries):
            colors[ancestry] = mcolors.to_hex(cmap_other(i))

    # Add a default color for (None, None) if it's not already handled
    if (None, None) not in colors:
        colors[(None, None)] = 'gray'

    return colors


def consolidate_ancestry_blocks(df: pd.DataFrame, chromatid_label: str) -> List[dict]:
    """
    Consolidates contiguous blocks of the same ancestry for a given chromatid.
    This version uses 'ancestry_chromatidX_pop' and 'ancestry_chromatidX_founder_id' columns.
    """
    blocks = []
    current_ancestry = None
    block_start_idx = None

    # Determine which columns to use based on chromatid_label
    if chromatid_label == 'chromatid1':
        pop_col = 'ancestry_chromatid1_pop'
        founder_col = 'ancestry_chromatid1_founder_id'
    elif chromatid_label == 'chromatid2':
        pop_col = 'ancestry_chromatid2_pop'
        founder_col = 'ancestry_chromatid2_founder_id'
    else:
        print(f"Error: Invalid chromatid label '{chromatid_label}'. Must be 'chromatid1' or 'chromatid2'.")
        return []

    # Ensure the necessary columns exist
    if pop_col not in df.columns or founder_col not in df.columns:
        print(f"Error: Required ancestry columns '{pop_col}' or '{founder_col}' not found in DataFrame.")
        return []

    # Sort by locus_plot_idx for correct block consolidation
    chromatid_df = df.sort_values(by='locus_plot_idx').reset_index(drop=True)

    for i, row in chromatid_df.iterrows():
        anc_pop = row[pop_col]
        founder_id = row[founder_col]

        # Handle NaN/None values for ancestry
        if pd.isna(anc_pop) or pd.isna(founder_id):
            processed_ancestry = (None, None)
        else:
            processed_ancestry = (str(anc_pop), int(founder_id)) # Ensure consistent types

        if block_start_idx is None:
            # Start of a new block
            current_ancestry = processed_ancestry
            block_start_idx = i
        elif processed_ancestry != current_ancestry:
            # End of current block, save it
            blocks.append({
                'start_idx': block_start_idx,
                'end_idx': i - 1,
                'ancestry': current_ancestry
            })
            # Start a new block
            current_ancestry = processed_ancestry
            block_start_idx = i

    # Add the last block
    if block_start_idx is not None:
        blocks.append({
            'start_idx': block_start_idx,
            'end_idx': len(chromatid_df) - 1,
            'ancestry': current_ancestry
        })
    return blocks

# --- MODIFIED plot_diploid_chromosome_ancestry function ---
def plot_diploid_chromosome_ancestry(
    individual_id: int,
    diploid_chr_id: int,
    generation_label: str,
    locus_data_df: pd.DataFrame,
    marker_height: float = 0.4, # Height of each chromatid bar
    spacing: float = 0.2,       # Vertical spacing between chromatids
    save_filename: Optional[str] = None,
    locus_width_unit: float = 0.1 # Width in inches for each locus to control overall plot width
) -> Optional[Set[Tuple[str, int]]]: # Now returns the set of used ancestries
    target_data = locus_data_df[
        (locus_data_df['individual_id'] == individual_id) &
        (locus_data_df['diploid_chr_id'] == diploid_chr_id) &
        (locus_data_df['generation'] == generation_label)
    ].sort_values(by='locus_position').reset_index(drop=True)

    if target_data.empty:
        print(f"No data found for Individual ID {individual_id}, Chromosome {diploid_chr_id} in generation {generation_label}.")
        return None # Return None if no data

    num_loci = len(target_data['locus_position'].unique()) # Use unique locus positions for count

    unique_ancestries = get_unique_ancestries(locus_data_df) # All possible ancestries
    ancestry_colors = generate_color_map_for_ancestries(unique_ancestries)

    if (None, None) not in ancestry_colors:
        ancestry_colors[(None, None)] = 'gray'

    # Calculate figure width based on total number of loci
    fig_width = max(10, num_loci * locus_width_unit)
    fig_height = 4.0 # Fixed height, as everything is on one row now

    # Create a single subplot
    fig, ax = plt.subplots(1, 1, figsize=(fig_width, fig_height))

    legend_elements = []
    # This set will store the (anc_pop, founder_id) that are actually present on the plotted chromosome
    present_founder_ancestries = set() 

    global_locus_start = target_data['locus_position'].iloc[0] if not target_data.empty else 0
    # Adjust locus indices to be 0-based for plotting convenience if they aren't already
    target_data['locus_plot_idx'] = target_data['locus_position'] - global_locus_start

    # Define vertical positions for chromatids
    chromatid_y_positions = {
        'chromatid1': marker_height + spacing,
        'chromatid2': 0.0
    }
    y_min_limit = -0.1
    y_max_limit = marker_height + spacing + marker_height + 0.1

    ax.set_title(f"Individual {individual_id}, Chromosome {diploid_chr_id} ({generation_label})\nAncestral Origin", fontsize=14)

    ax.set_ylabel("") # No general Y-axis label needed
    ax.set_yticks([]) # Hide default y-ticks
    ax.set_yticklabels([]) # Hide default y-tick labels

    # X-axis setup for the *entire* chromosome
    # Use the 0-based plot indices for ticks
    max_plot_idx = target_data['locus_plot_idx'].max()
    tick_interval = max(1, int(np.ceil(max_plot_idx / 20))) # Aim for about 20 ticks
    
    # Generate ticks for the plot indices
    ticks_plot_idx = np.arange(0, max_plot_idx + 1, tick_interval)
    # Map these plot indices back to original locus positions for labels
    labels_original_locus = [int(idx + global_locus_start) for idx in ticks_plot_idx]

    ax.set_xticks(ticks_plot_idx)
    ax.set_xticklabels(labels_original_locus, rotation=45, ha='right')
    ax.set_xlabel("Locus Position", fontsize=12)

    ax.set_xlim(-0.5, max_plot_idx + 0.5) # Extend slightly beyond the last locus
    ax.set_ylim(y_min_limit, y_max_limit)
    ax.set_aspect('auto', adjustable='box')


    # Draw horizontal lines for chromatids (these will be the base lines)
    for chromatid_label_key, y_base in chromatid_y_positions.items():
        ax.plot([-0.5, max_plot_idx + 0.5], [y_base, y_base], color='black', linewidth=1.5, zorder=1)
        ax.plot([-0.5, max_plot_idx + 0.5], [y_base + marker_height, y_base + marker_height], color='black', linewidth=1.5, zorder=1)

        # Labels for chromatids
        transform_x_axes_y_data = blended_transform_factory(ax.transAxes, ax.transData)
        chromatid_display_name = 'Chromatid 1' if chromatid_label_key == 'chromatid1' else 'Chromatid 2'
        ax.text(-0.02, y_base + marker_height/2, chromatid_display_name, va='center', ha='right', fontsize=10, transform=transform_x_axes_y_data)


    # Plot ancestry blocks for each chromatid
    for chromatid_label, y_bottom_pos in chromatid_y_positions.items():
        blocks = consolidate_ancestry_blocks(target_data, chromatid_label)

        for block in blocks:
            block_start_local_idx = block['start_idx']
            block_end_local_idx = block['end_idx']
            ancestry = block['ancestry']

            if ancestry == (None, None):
                display_label = "Missing Data"
                color = ancestry_colors.get((None, None), 'gray')
                # No text for missing data
                continue # Skip to next block if missing data
            else:
                anc_pop, founder_id = ancestry
                display_label = f'{anc_pop}_{founder_id}' # This will be P_A_1, P_B_2 etc.
                color = ancestry_colors.get(ancestry, 'gray')
                
                # Add to the set of present founder ancestries
                present_founder_ancestries.add(ancestry)


            rect_width = (block_end_local_idx - block_start_local_idx + 1)
            rect_x = block_start_local_idx - 0.5

            rect = patches.Rectangle(
                (rect_x, y_bottom_pos),
                rect_width,
                marker_height,
                facecolor=color,
                edgecolor='none',
                linewidth=0,
                zorder=2
            )
            ax.add_patch(rect)

            # --- MODIFIED TEXT PLOTTING LOGIC (from previous interaction) ---
            text_to_display = ""
            font_size = 7 # Default font size

            # Always display founder_id for P_A or P_B, or if it's a very long segment
            if anc_pop in ['P_A', 'P_B']:
                text_to_display = str(founder_id)
                # Make font larger for P_A/P_B if possible, or if the segment is large
                if rect_width >= 3: # If block is 3 loci or more
                    font_size = 8
                elif rect_width == 2: # If block is 2 loci
                    font_size = 7
                else: # Very small blocks might still get text if it's P_A/P_B
                    font_size = 6
            elif rect_width > 5: # For other ancestries, only show ID if segment is long enough
                text_to_display = str(founder_id)
                font_size = 7 # Keep default size

            if text_to_display: # Only plot text if there's something to display
                text_x_pos = block_start_local_idx + (rect_width / 2) - 0.5
                try:
                    rgb = mcolors.to_rgb(color)
                    text_color = 'black' if np.mean(rgb) > 0.5 else 'white'
                except Exception:
                    text_color = 'black'

                if rect_width >= 1: # Always try to plot text if rect_width is at least 1
                    ax.text(
                        text_x_pos,
                        y_bottom_pos + marker_height / 2,
                        text_to_display,
                        ha='center',
                        va='center',
                        fontsize=font_size,
                        color=text_color,
                        zorder=3
                    )
            # --- END MODIFIED TEXT PLOTTING LOGIC ---

            if display_label not in [le.get_label() for le in legend_elements]: # Check against existing labels
                legend_elements.append(Line2D([0], [0], marker='s', color='w', label=display_label,
                                              markerfacecolor=color, markersize=8))

    if legend_elements:
        legend_elements.sort(key=lambda x: x.get_label())

        fig.legend(
            handles=legend_elements,
            title="Ancestral Origin",
            loc='center right',
            bbox_to_anchor=(0.98, 0.5),
            borderaxespad=0.0,
            frameon=False,
            fontsize=9,
            title_fontsize=10
        )

    fig.subplots_adjust(left=0.05, right=0.75, top=0.85, bottom=0.15)

    if save_filename:
        plt.savefig(save_filename, bbox_inches='tight', dpi=300)
        print(f"Chromosome plot saved to: {save_filename}")
        plt.close(fig)
    else:
        plt.show()
    
    return present_founder_ancestries # Return the set of ancestries present on the chromosome


# --- MODIFIED plot_pedigree_for_individual function ---
def plot_pedigree_for_individual(
    locus_level_df: pd.DataFrame,
    target_individual_id: int,
    ancestry_colors: dict, # Pass the global ancestry_colors map
    present_founder_ancestries_on_chr: Set[Tuple[str, int]], # NEW: set of founders actually present on chromosome
    output_directory: str = r"C:\Users\sophi\Jupyter_projects\Hybrid_Code\output_data\plots\pedigree_visuals"
):
    """
    Creates and saves a visual pedigree tree for a specific individual,
    with founders colored based on their ancestral origin and ID,
    but ONLY if they are present in the corresponding chromosome visualization.
    """
    if target_individual_id not in locus_level_df['individual_id'].values:
        print(f"Error: Target individual ID {target_individual_id} not found in the data.")
        return

    pedigree_df = locus_level_df[['individual_id', 'parent1_id', 'parent2_id', 'generation',
                                 'ancestry_chromatid1_pop', 'ancestry_chromatid1_founder_id',
                                 'ancestry_chromatid2_pop', 'ancestry_chromatid2_founder_id']].drop_duplicates().copy()

    G = nx.DiGraph()

    for _, row in pedigree_df.iterrows():
        child = row['individual_id']
        parent1 = row['parent1_id']
        parent2 = row['parent2_id']
        generation = row['generation']

        G.add_node(child, shape='circle', label=f"ID: {int(child)}\nGen: {generation}")

        # Helper to add parent node with conditional coloring
        def add_parent_node(parent_id, child_id):
            if pd.notna(parent_id):
                parent_id_int = int(parent_id)
                parent_info = pedigree_df[pedigree_df['individual_id'] == parent_id_int].iloc[0] if parent_id_int in pedigree_df['individual_id'].values else None
                
                node_label = f"ID: {int(parent_id_int)}"
                node_fillcolor = 'lightgray' # Default for non-founders or founders not on chromosome
                node_fontcolor = 'black'
                node_style = 'filled'
                node_shape = 'square'

                if parent_info is not None and parent_info['generation'] in ['P_A', 'P_B']:
                    current_ancestry_tuple = (str(parent_info['generation']), parent_id_int)
                    
                    if current_ancestry_tuple in present_founder_ancestries_on_chr:
                        # Only color if this founder is present in the chromosome visualization
                        if current_ancestry_tuple in ancestry_colors:
                            node_fillcolor = ancestry_colors[current_ancestry_tuple]
                            try: # Determine text color for contrast
                                rgb = mcolors.to_rgb(node_fillcolor)
                                node_fontcolor = 'black' if np.mean(rgb) > 0.5 else 'white'
                            except Exception:
                                node_fontcolor = 'black'
                        node_label += f"\nGen: {parent_info['generation']} ({parent_id_int})"
                    else:
                        # If P_A/P_B founder is NOT present on chromosome, make it a distinct gray/outline
                        node_fillcolor = 'grey'
                        node_style = 'solid' # Just outline, or 'filled' with a very light grey
                        node_fontcolor = 'black' # Default text color
                        node_label += f"\nGen: {parent_info['generation']} ({parent_id_int})" # Still label them

                elif parent_info is not None: # For F1, F2 etc. parents
                    node_label += f"\nGen: {parent_info['generation']}"
                    
                G.add_node(parent_id_int, shape=node_shape, label=node_label,
                           fillcolor=node_fillcolor, style=node_style, fontcolor=node_fontcolor)
                G.add_edge(parent_id_int, child_id)

        add_parent_node(parent1, child)
        add_parent_node(parent2, child)

    ancestors = nx.ancestors(G, target_individual_id)
    subgraph_nodes = list(ancestors) + [target_individual_id]
    
    # Also include direct parents if they weren't already pulled in as ancestors
    target_row = pedigree_df[pedigree_df['individual_id'] == target_individual_id]
    if not target_row.empty:
        p1 = target_row['parent1_id'].iloc[0]
        p2 = target_row['parent2_id'].iloc[0]
        if pd.notna(p1): subgraph_nodes.append(int(p1))
        if pd.notna(p2): subgraph_nodes.append(int(p2))
    
    subgraph_nodes = list(set(subgraph_nodes)) # Remove duplicates

    pedigree_subgraph = G.subgraph(subgraph_nodes).copy()

    pedigree_graph = nx.drawing.nx_pydot.to_pydot(pedigree_subgraph)
    pedigree_graph.set_rankdir("TB")

    target_node = pedigree_graph.get_node(str(target_individual_id))
    if target_node:
        target_node[0].set_color("red")
        target_node[0].set_style("filled")
        target_node[0].set_fillcolor("red")
        target_node[0].set_fontcolor("white")

    if not os.path.exists(output_directory):
        os.makedirs(output_directory, exist_ok=True)

    output_filename = f"pedigree_ID_{target_individual_id}_filtered_colored.png" # Changed filename
    output_path = os.path.join(output_directory, output_filename)
    
    try:
        pedigree_graph.write_png(output_path)
        print(f"Pedigree plot saved to: {output_path}") # Explicit print
    except Exception as e:
        print(f"Error saving pedigree plot: {e}")
        print("Ensure Graphviz is installed and 'dot.exe' is in your system PATH or correctly set.")


# --- Main Script Execution ---

# Your file path
file_path = r"C:\Users\sophi\Jupyter_projects\Hybrid_Code\output_data\datafr\locus_level_genotypes_with_ancestry_pedigree_20.csv"

# Load the full DataFrame
locus_level_df = pd.read_csv(file_path)

# --- Function to print available IDs by generation (kept for completeness) ---
def print_available_ids_by_generation(file_path: str):
    """
    Loads the locus-level data and prints a list of unique individual IDs
    available for each generation.
    """
    if not os.path.exists(file_path):
        print(f"Error: File not found at '{file_path}'. Please check the path.")
        return
    full_df = pd.read_csv(file_path)
    individuals = full_df[['generation', 'individual_id']].drop_duplicates()
    print("--- Available Individual IDs per Generation ---")
    grouped_by_gen = individuals.groupby('generation')['individual_id'].apply(list)
    for gen, ids in grouped_by_gen.items():
        print(f"\nGeneration: {gen} (Count: {len(ids)})")
        print(ids)

print_available_ids_by_generation(file_path)

# --- Directories for plots ---
plots_directory = r"C:\Users\sophi\Jupyter_projects\Hybrid_Code\output_data\plots\pedigree_visuals"
os.makedirs(plots_directory, exist_ok=True) # Ensure the directory exists

# --- Configuration for the specific individual to plot ---
specific_id_to_plot = 78 # Example: Individual F4_38
specific_gen_for_id = 'F5'
specific_chr_to_plot = 2

# First, plot the chromosome and get the *actual* founders present
print(f"\nPlotting specific individual (ID: {specific_id_to_plot}) from generation '{specific_gen_for_id}', Chromosome {specific_chr_to_plot} with merged blocks.")
plot_save_path_specific_ind = os.path.join(
    plots_directory,
    f"specific_ind_{specific_id_to_plot}_gen_{specific_gen_for_id}_chr_{specific_chr_to_plot}_ancestry_merged.png"
)

# Capture the set of present founder ancestries from the chromosome plot
present_founders = plot_diploid_chromosome_ancestry(
    individual_id=specific_id_to_plot,
    diploid_chr_id=specific_chr_to_plot,
    generation_label=specific_gen_for_id,
    locus_data_df=locus_level_df,
    save_filename=plot_save_path_specific_ind,
    locus_width_unit=0.1
)

# Now, plot the pedigree, passing the collected founders and the global color map
if present_founders is not None:
    # Need to generate the full ancestry_colors map first for the pedigree,
    # making sure it's consistent with the chromosome plot's color generation.
    all_unique_ancestries_global = get_unique_ancestries(locus_level_df)
    ancestry_colors_global = generate_color_map_for_ancestries(all_unique_ancestries_global)

    plot_pedigree_for_individual(
        locus_level_df,
        target_individual_id=specific_id_to_plot,
        ancestry_colors=ancestry_colors_global, # Pass the full color map
        present_founder_ancestries_on_chr=present_founders, # Pass the filtered set
        output_directory=plots_directory
    )
else:
    print(f"Could not generate pedigree for ID {specific_id_to_plot} as chromosome data was not found or plot failed.")

--- Available Individual IDs per Generation ---

Generation: F1 (Count: 20)
[41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60]

Generation: F2 (Count: 10)
[61, 62, 63, 64, 65, 66, 67, 68, 69, 70]

Generation: F3 (Count: 5)
[71, 72, 73, 74, 75]

Generation: F4 (Count: 2)
[76, 77]

Generation: F5 (Count: 1)
[78]

Generation: P_A (Count: 20)
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

Generation: P_B (Count: 20)
[21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40]

Plotting specific individual (ID: 78) from generation 'F5', Chromosome 2 with merged blocks.
Chromosome plot saved to: C:\Users\sophi\Jupyter_projects\Hybrid_Code\output_data\plots\pedigree_visuals\specific_ind_78_gen_F5_chr_2_ancestry_merged.png
Pedigree plot saved to: C:\Users\sophi\Jupyter_projects\Hybrid_Code\output_data\plots\pedigree_visuals\pedigree_ID_78_filtered_colored.png
